In [1]:
import numpy as np
import requests

import requests
from datetime import datetime, timedelta
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
import csv
import time

#1. Getting prices
def prices(startd,endd,tic):
    info=[]
    key='&apikey=ZKMMTO1ATDBLXH2K'
    ticker='&symbol='+str(tic)
#    endpoint='function=TIME_SERIES_MONTHLY_ADJUSTED'
    endpoint='function=TIME_SERIES_DAILY_ADJUSTED'
    size='&outputsize=full'
    web='https://www.alphavantage.co/query?'
    url =web+endpoint+ticker+size+key

    r = requests.get(url)
    if r.status_code==200:
        print('connection successful')
        data = r.json() #need to convert to json to navigate
        r1=data.get('Time Series (Daily)', {})
        #r1=data.get('Monthly Adjusted Time Series', {})
        r2=data['Time Series (Daily)']
        #r2=data['Monthly Adjusted Time Series']
        
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info
    

    

def compute_log_returns(data, freq='daily'):
    """
    Computes log returns at specified frequency.
    
    Parameters:
    -----------
    data : list of lists
        Price data: [[ticker, date, adjusted_close], ...]
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    
    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return], ...]
    """
    sorted_data = sorted(data, key=lambda x: x[1])  # Sort by date (index 1)
    data_with_logret = []
    
    if freq == 'daily':
        for i in range(len(sorted_data)):
            if i == 0:
                data_with_logret.append(sorted_data[i] + [None])
            else:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[i-1][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
    
    elif freq == 'weekly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 7:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    elif freq == 'monthly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 30:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    elif freq == 'yearly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 360:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    return data_with_logret


#6. Download, compute returns, and export to CSV
def download_and_export(ticker, startd, endd, freq='daily', path=None):
    """
    Downloads price data, computes log and simple returns, and exports to CSV.

    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g. 'NVDA')
    startd : str
        Start date 'YYYY-MM-DD'
    endd : str
        End date 'YYYY-MM-DD'
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    path : str
        File path for CSV output. Defaults to '{ticker}_returns.csv'

    Returns:
    --------
    list of lists: [[ticker, date, price, log_return, simple_return], ...]
    First row is a header row.
    """
    if path is None:
        path = f"{ticker}_returns.csv"

    raw = prices(startd, endd, ticker)
    with_log = compute_log_returns(raw, freq=freq)
    with_both = compute_returns(with_log)

    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        for row in with_both:
            writer.writerow(row)

    print(f"Exported {len(with_both)-1} rows to {path}")
    return with_both


In [3]:
def compute_returns(data):
    """
    Computes simple and log returns from compute_log_returns output.
    
    Parameters:
    -----------
    data : list of lists
        [[ticker, date, adjusted_close, log_return], ...]
    
    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return, simple_return], ...]
    First row is a header row.
    """
    header = ["ticker", "date", "price", "log_return", "simple_return"]
    result = [header]
    for i in range(len(data)):
        if i == 0 or data[i-1][2] is None:
            result.append(data[i][:4] + [None])
        else:
            price_current = float(data[i][2])
            price_previous = float(data[i-1][2])
            simple_ret = (price_current - price_previous) / price_previous
            result.append(data[i][:4] + [simple_ret])
    return result


In [2]:
data_nvda = prices("2022-01-01", "2025-12-31", "NVDA")
time.sleep(5)
data_jpm = prices("2022-01-01", "2025-12-31", "JPM")
time.sleep(5)
data_aapl = prices("2022-01-01", "2025-12-31", "AAPL")
time.sleep(5)
data_spy = prices("2022-01-01", "2025-12-31", "SPY")

connection successful
connection successful
connection successful
connection successful


In [4]:
daily_nvda = compute_log_returns(data_nvda, freq='daily')
daily_jpm = compute_log_returns(data_jpm, freq='daily')
daily_aapl = compute_log_returns(data_aapl, freq='daily')
daily_spy = compute_log_returns(data_spy, freq='daily')

In [6]:
nvda = compute_returns(daily_nvda)
jpm = compute_returns(daily_jpm)
aapl = compute_returns(daily_aapl)
spy = compute_returns(daily_spy)

In [7]:
import os
os.makedirs("Group_Proj_CSVs", exist_ok=True)

nvda = download_and_export("NVDA", "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/nvda_returns.csv")
time.sleep(5)
jpm  = download_and_export("JPM",  "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/jpm_returns.csv")
time.sleep(5)
aapl = download_and_export("AAPL", "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/aapl_returns.csv")
time.sleep(5)
spy  = download_and_export("SPY",  "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/spy_returns.csv")

connection successful
Exported 1003 rows to Group_Proj_CSVs/nvda_returns.csv
connection successful
Exported 1003 rows to Group_Proj_CSVs/jpm_returns.csv
connection successful
Exported 1003 rows to Group_Proj_CSVs/aapl_returns.csv
connection successful
Exported 1003 rows to Group_Proj_CSVs/spy_returns.csv


In [31]:
def rsi_full(data, period=14):
    df = pd.DataFrame(data[1:], columns=data[0])
    df['simple_return'] = pd.to_numeric(df['simple_return'], errors='coerce')
    df = df.dropna(subset=['simple_return'])
    
    gains = df['simple_return'].apply(lambda x: x if x > 0 else 0)
    losses = df['simple_return'].apply(lambda x: abs(x) if x < 0 else 0)
    
    avg_gain = gains.rolling(window=period).mean()
    avg_loss = losses.rolling(window=period).mean()
    
    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    df['signal'] = 0
    df.loc[df['RSI'] < 30, 'signal'] = 1
    df.loc[df['RSI'] > 70, 'signal'] = -1
    
    return df

spy_full  = rsi_full(spy)
nvda_full = rsi_full(nvda)
jpm_full  = rsi_full(jpm)
aapl_full = rsi_full(aapl)

In [32]:
spy_signals  = spy_full[spy_full['signal'] != 0][['date', 'price', 'log_return', 'simple_return', 'RSI', 'signal']]
nvda_signals = nvda_full[nvda_full['signal'] != 0][['date', 'price', 'log_return', 'simple_return', 'RSI', 'signal']]
jpm_signals  = jpm_full[jpm_full['signal'] != 0][['date', 'price', 'log_return', 'simple_return', 'RSI', 'signal']]
aapl_signals = aapl_full[aapl_full['signal'] != 0][['date', 'price', 'log_return', 'simple_return', 'RSI', 'signal']]

In [33]:
def execute_trade(df, ticker_name):
    trade_returns = []
    
    for _, row in df.iterrows():
        signal = row['signal']
        if signal != 0:
            rec = recommendation(signal, row['RSI'])
            log_ret = float(row['log_return'])
            trade_returns.append({
                'date': row['date'],
                'ticker': ticker_name,
                'recommendation': rec,
                'log_return': log_ret
            })
    
    return trade_returns

# Run for each ticker
spy_trades  = execute_trade(spy_signals,  'SPY')
nvda_trades = execute_trade(nvda_signals, 'NVDA')
jpm_trades  = execute_trade(jpm_signals,  'JPM')
aapl_trades = execute_trade(aapl_signals, 'AAPL')

all_trade_returns = (
    [t['log_return'] for t in spy_trades] +
    [t['log_return'] for t in nvda_trades] +
    [t['log_return'] for t in jpm_trades] +
    [t['log_return'] for t in aapl_trades]
)


In [34]:
def compute_statistics(all_trades, spy_trades, nvda_trades, jpm_trades, aapl_trades, risk_free_rate=0.04):
    
    df = pd.DataFrame(all_trades)
    df['date'] = pd.to_datetime(df['date'])
    df['log_return'] = df['log_return'].astype(float)
    
    returns = df['log_return'].values
    long_returns  = df[df['recommendation'].isin(['buy', 'strong buy'])]['log_return'].values
    short_returns = df[df['recommendation'].isin(['sell', 'strong sell'])]['log_return'].values
    
    # a. Number of trades per month
    df['month'] = df['date'].dt.to_period('M')
    trades_per_month = df.groupby('month').size()
    
    # b. Average return and statistical significance
    avg_return = returns.mean()
    t_stat, p_value = stats.ttest_1samp(returns, 0)
    
    # c. Average return for longs
    avg_long = long_returns.mean() if len(long_returns) > 0 else None
    
    # d. Average return for shorts
    avg_short = short_returns.mean() if len(short_returns) > 0 else None
    
    # e. Cumulative return annualized (equal weighted across 4 tickers)
    cum_log_returns = []
    for trades in [spy_trades, nvda_trades, jpm_trades, aapl_trades]:
        df_t = pd.DataFrame(trades)
        if len(df_t) > 0:
            cum_log_returns.append(df_t['log_return'].astype(float).sum())
    portfolio_cum = np.exp(np.mean(cum_log_returns)) - 1
    ann_cum_return = (1 + portfolio_cum) ** (365 / (365 * 3)) - 1

    # f. Sharpe Ratio annualized
    daily_rf = risk_free_rate / 252
    excess_returns = returns - daily_rf
    sharpe = (excess_returns.mean() / excess_returns.std()) * np.sqrt(252)
    
    # g. Sortino Ratio annualized
    downside = returns[returns < 0]
    downside_std = downside.std() if len(downside) > 0 else np.nan
    sortino = (avg_return - daily_rf) / downside_std * np.sqrt(252)
    
    # h. Jensen's Alpha
    spy_df = pd.DataFrame(spy_trades)
    spy_df['date'] = pd.to_datetime(spy_df['date'])
    merged = df.merge(spy_df[['date', 'log_return']], on='date', suffixes=('_port', '_mkt'))
    if len(merged) > 1:
        port_r = merged['log_return_port'].values
        mkt_r  = merged['log_return_mkt'].values
        beta   = np.cov(port_r, mkt_r)[0,1] / np.var(mkt_r)
        alpha  = avg_return - (daily_rf + beta * (mkt_r.mean() - daily_rf))
        ann_alpha = alpha * 252
    else:
        ann_alpha, beta = np.nan, np.nan
    
    # i. VaR at 5%
    var_5 = np.percentile(returns, 5)
    
    # Print results
    print("=" * 50)
    print("TRADING STRATEGY STATISTICS")
    print("=" * 50)
    print(f"\na. Trades per month:\n{trades_per_month.to_string()}")
    print(f"\nb. Average log-return:     {avg_return:.6f}")
    print(f"   T-stat:                 {t_stat:.4f}")
    print(f"   P-value:                {p_value:.4f} {'(significant)' if p_value < 0.05 else '(not significant)'}")
    print(f"\nc. Avg return (longs):     {avg_long:.6f}" if avg_long is not None else "\nc. No long trades")
    print(f"\nd. Avg return (shorts):    {avg_short:.6f}" if avg_short is not None else "\nd. No short trades")
    print(f"\ne. Cumulative return (ann):{ann_cum_return:.4%}")
    print(f"\nf. Sharpe Ratio (ann):     {sharpe:.4f}")
    print(f"\ng. Sortino Ratio (ann):    {sortino:.4f}")
    print(f"\nh. Jensen's Alpha (ann):   {ann_alpha:.6f}")
    print(f"   Beta:                   {beta:.4f}")
    print(f"\ni. VaR (5%):               {var_5:.6f}")
    print("=" * 50)

# Usage
all_trades = spy_trades + nvda_trades + jpm_trades + aapl_trades
compute_statistics(all_trades, spy_trades, nvda_trades, jpm_trades, aapl_trades)

TRADING STRATEGY STATISTICS

a. Trades per month:
month
2022-01    19
2022-02     7
2022-03    27
2022-04    30
2022-05     5
2022-06    19
2022-07    14
2022-08    36
2022-09    36
2022-10     9
2022-11    32
2022-12     8
2023-01    33
2023-02    19
2023-03    17
2023-04    25
2023-05     9
2023-06    37
2023-07    31
2023-08    31
2023-09    21
2023-10    15
2023-11    44
2023-12    37
2024-01    36
2024-02    36
2024-03    34
2024-04     9
2024-05    46
2024-06    34
2024-07    29
2024-08    29
2024-09    15
2024-10    24
2024-11    13
2024-12    27
2025-01    24
2025-02    18
2025-03    31
2025-04    15
2025-05    46
2025-06    15
2025-07    51
2025-08    12
2025-09    19
2025-10    12
2025-11    15
2025-12     8
Freq: M

b. Average log-return:     0.002611
   T-stat:                 4.2812
   P-value:                0.0000 (significant)

c. Avg return (longs):     -0.008692

d. Avg return (shorts):    0.006488

e. Cumulative return (ann):28.6875%

f. Sharpe Ratio (ann):     1.875